In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score

# ---- Config ----
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 5
EPOCHS_FT = 5
AUTOTUNE = tf.data.AUTOTUNE

# ✅ Just set your root here (no file reading)
ROOT = "/path/to/local/resource"

tf.random.set_seed(SEED)
np.random.seed(SEED)

print("Using ROOT:", ROOT)


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# Expected LC25000 lung classes
EXPECTED_FOLDER_CLASSES = {"lung_aca", "lung_n", "lung_scc"}
PREFIX_MAP = {"lungaca": "lung_aca", "lungn": "lung_n", "lungscc": "lung_scc"}

def is_image(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def detect_folder_layout(root: str) -> bool:
    rootp = Path(root)
    if not rootp.exists():
        raise FileNotFoundError(f"ROOT not found: {root}")
    # If immediate subfolders contain images, treat as ImageFolder-like
    for d in rootp.iterdir():
        if d.is_dir():
            if any(f.is_file() and is_image(f) for f in d.iterdir()):
                return True
    return False

def scan_dataset(root: str) -> pd.DataFrame:
    rootp = Path(root)
    folder_layout = detect_folder_layout(root)

    rows = []
    if folder_layout:
        # Label = folder name (only keep expected lung_* dirs; if others exist, you can remove the filter)
        for class_dir in rootp.iterdir():
            if not class_dir.is_dir():
                continue
            label = class_dir.name
            if label not in EXPECTED_FOLDER_CLASSES:
                continue
            for imgp in class_dir.rglob("*"):
                if imgp.is_file() and is_image(imgp):
                    rows.append((str(imgp), label))
    else:
        # Label from filename prefix anywhere under root
        for imgp in rootp.rglob("*"):
            if not (imgp.is_file() and is_image(imgp)):
                continue
            fn = imgp.name.lower()
            label = None
            for pref, lbl in PREFIX_MAP.items():
                if fn.startswith(pref):
                    label = lbl
                    break
            if label is not None:
                rows.append((str(imgp), label))

    df = pd.DataFrame(rows, columns=["filepath", "label"])
    if df.empty:
        raise RuntimeError(
            "No images found. Check ROOT, or ensure filenames start with lungaca/lungn/lungscc, "
            "or that class subfolders exist (lung_aca, lung_n, lung_scc)."
        )

    return df

df = scan_dataset(ROOT)
df.head(), df["label"].value_counts()


In [ ]:
labels = sorted(df["label"].unique().tolist())
label_to_idx = {lbl: i for i, lbl in enumerate(labels)}
idx_to_label = {i: lbl for lbl, i in label_to_idx.items()}

df["y"] = df["label"].map(label_to_idx).astype(int)

print("Found classes:", labels)
print(df["label"].value_counts())

# If you expected 3 classes but got 2, this will tell you.
if len(labels) < 3:
    print("\n⚠️ Note: fewer than 3 lung classes found. If you want 3-class (aca/n/scc), verify folders/files.")


In [ ]:
train_size, val_size, test_size = 0.8, 0.1, 0.1
assert abs(train_size + val_size + test_size - 1.0) < 1e-9

y = df["y"].values
idx = np.arange(len(df))

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=(1.0 - train_size), random_state=SEED)
train_idx, temp_idx = next(sss1.split(idx, y))

# temp -> val/test
y_temp = y[temp_idx]
val_frac_of_temp = val_size / (val_size + test_size)
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=(1.0 - val_frac_of_temp), random_state=SEED)
val_rel, test_rel = next(sss2.split(np.arange(len(temp_idx)), y_temp))

val_idx = temp_idx[val_rel]
test_idx = temp_idx[test_rel]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

print("Sizes:", len(train_df), len(val_df), len(test_df))
print("Train dist:\n", train_df["label"].value_counts())


In [ ]:
# Choose a pretrained backbone + its preprocess_input
BACKBONE = "EfficientNetB0"
preprocess_input = tf.keras.applications.efficientnet.preprocess_input

def load_image(path, y):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)  # LC25000 is typically jpg
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, y

data_augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomFlip("vertical"),
    layers.RandomRotation(0.08),
])

def preprocess_train(img, y):
    img = tf.cast(img, tf.float32)
    img = data_augment(img, training=True)
    img = preprocess_input(img)
    return img, y

def preprocess_eval(img, y):
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img, y

def make_ds(frame: pd.DataFrame, training: bool):
    paths = frame["filepath"].values
    ys = frame["y"].values
    ds = tf.data.Dataset.from_tensor_slices((paths, ys))
    if training:
        ds = ds.shuffle(buffer_size=len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.map(preprocess_train if training else preprocess_eval, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df, training=False)
test_ds  = make_ds(test_df, training=False)


In [ ]:
num_classes = len(labels)

base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.ModelCheckpoint("best_lung_tf.keras", monitor="val_accuracy", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
]


In [ ]:
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)


In [ ]:
# Unfreeze backbone
base.trainable = True

# Optional: only fine-tune the top N layers
# for layer in base.layers[:-40]:
#     layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    callbacks=callbacks
)


In [ ]:
# Load best checkpoint (if saved)
best_model = keras.models.load_model("best_lung_tf.keras")

# Predict
y_true = test_df["y"].values
probs = best_model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
cm = confusion_matrix(y_true, y_pred)

print("Test Accuracy:", acc)
print("Test Macro-F1:", macro_f1)
print("\nConfusion Matrix (rows=true, cols=pred):\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=[idx_to_label[i] for i in range(len(labels))], digits=4))


In [ ]:
import json

artifacts = {
    "labels": labels,
    "label_to_idx": label_to_idx,
    "idx_to_label": idx_to_label,
    "img_size": IMG_SIZE,
    "backbone": BACKBONE
}

Path("artifacts").mkdir(exist_ok=True)
Path("artifacts/label_map.json").write_text(json.dumps(artifacts, indent=2))

best_model.save("artifacts/final_lung_model.keras")

print("Saved artifacts/label_map.json and artifacts/final_lung_model.keras")


In [ ]:
import numpy as np

# True labels (must match test_ds order; your pipeline uses shuffle=False for eval)
y_true = test_df["y"].values

# Model loss/accuracy on test set
test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print("Test loss:", test_loss)
print("Test acc :", test_acc)

# Probabilities + predicted class
probs = best_model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)

print("probs shape:", probs.shape, "num_classes:", probs.shape[1])
print("y_true shape:", y_true.shape, "y_pred shape:", y_pred.shape)


In [ ]:
from sklearn.metrics import (
    precision_recall_fscore_support,
    classification_report,
    accuracy_score
)

acc = accuracy_score(y_true, y_pred)

prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)
prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

prec_c, rec_c, f1_c, support_c = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

print("Accuracy       :", acc)
print("Macro Precision:", prec_macro)
print("Macro Recall   :", rec_macro)
print("Macro F1       :", f1_macro)
print("Weighted Prec  :", prec_weighted)
print("Weighted Recall:", rec_weighted)
print("Weighted F1    :", f1_weighted)

target_names = [idx_to_label[i] for i in range(len(labels))]
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=target_names, digits=4, zero_division=0))


In [ ]:
from sklearn.metrics import roc_auc_score

num_classes = probs.shape[1]
y_true_1hot = np.eye(num_classes)[y_true]

# Overall AUC (one-vs-rest)
auc_macro = roc_auc_score(y_true_1hot, probs, multi_class="ovr", average="macro")
auc_weighted = roc_auc_score(y_true_1hot, probs, multi_class="ovr", average="weighted")

# Per-class AUC (OvR)
auc_per_class = []
for k in range(num_classes):
    auc_k = roc_auc_score(y_true_1hot[:, k], probs[:, k])
    auc_per_class.append(auc_k)

print("AUC macro   :", auc_macro)
print("AUC weighted:", auc_weighted)
for i, v in enumerate(auc_per_class):
    print(f"AUC {idx_to_label[i]}:", v)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig = plt.figure()
plt.imshow(cm , cmap="Blues")  # default colormap
plt.title("Confusion Matrix (Counts)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(len(target_names)), target_names, rotation=45, ha="right")
plt.yticks(range(len(target_names)), target_names)

# annotate
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.show()

fig = plt.figure()
plt.imshow(cm_norm , cmap="Blues")
plt.title("Confusion Matrix (Normalized)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(len(target_names)), target_names, rotation=45, ha="right")
plt.yticks(range(len(target_names)), target_names)

# annotate
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        plt.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure()
for k in range(num_classes):
    fpr, tpr, _ = roc_curve(y_true_1hot[:, k], probs[:, k])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{idx_to_label[k]} (AUC={roc_auc:.3f})")

# Chance line
plt.plot([0, 1], [0, 1], linestyle="--")

plt.title("ROC Curves (One-vs-Rest)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure()
for k in range(num_classes):
    prec, rec, _ = precision_recall_curve(y_true_1hot[:, k], probs[:, k])
    ap = average_precision_score(y_true_1hot[:, k], probs[:, k])
    plt.plot(rec, prec, label=f"{idx_to_label[k]} (AP={ap:.3f})")

plt.title("Precision–Recall Curves (One-vs-Rest)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(num_classes)
width = 0.25

plt.figure()
plt.bar(x - width, prec_c, width, label="Precision")
plt.bar(x,         rec_c,  width, label="Recall")
plt.bar(x + width, f1_c,   width, label="F1")

plt.xticks(x, target_names, rotation=45, ha="right")
plt.title("Per-class Precision / Recall / F1")
plt.ylim(0, 1.0)
plt.legend()
plt.grid(True, axis="y")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def concat_history(h1, h2):
    out = {}
    for k in h1.history.keys():
        out[k] = list(h1.history.get(k, [])) + list(h2.history.get(k, []))
    return out

if "history_head" in globals() and "history_ft" in globals():
    H = concat_history(history_head, history_ft)

    plt.figure()
    plt.plot(H.get("loss", []), label="train_loss")
    plt.plot(H.get("val_loss", []), label="val_loss")
    plt.title("Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure()
    plt.plot(H.get("accuracy", []), label="train_acc")
    plt.plot(H.get("val_accuracy", []), label="val_acc")
    plt.title("Accuracy Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("history_head / history_ft not found in globals(). Skip this cell.")
